In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("data/Data_with_coding(full).csv")

In [4]:
# List of variables of interest (benefit/level vars)
vars_list = [
    "CONCERNCO_W49_num","BENEFITCO_W49_num","POSNEGCO_W49_num",
    "TRACKCO1a_W49_num","TRACKCO1b_W49_num","CONTROLCO_W49_num","UNDERSTANDCO_W49_num","ANONYMOUS1CO_W49_num",
    "CONCERNGOV_W49_num","BENEFITGOV_W49_num","POSNEGGOV_W49_num",
    "TRACKGOV1a_W49_num","TRACKGOV1b_W49_num","CONTROLGOV_W49_num","UNDERSTANDGOV_W49_num","ANONYMOUS1GOV_W49_num"
]

In [5]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

# --- CONFIG ---
DATA_PATH = "data/Data_with_coding(full).csv"   # change to your file with respondent rows
OUTPUT_DIR = "qa_missingness_analysis"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Put your selected question IDs here (use either raw or _num recoded columns).
# Example: the 16 private/public surveillance items (numeric recodes)
selected_cols = [
    "CONCERNCO_W49_num","BENEFITCO_W49_num","POSNEGCO_W49_num",
    "TRACKCO1a_W49_num","TRACKCO1b_W49_num","CONTROLCO_W49_num","UNDERSTANDCO_W49_num","ANONYMOUS1CO_W49_num",
    "CONCERNGOV_W49_num","BENEFITGOV_W49_num","POSNEGGOV_W49_num",
    "TRACKGOV1a_W49_num","TRACKGOV1b_W49_num","CONTROLGOV_W49_num","UNDERSTANDGOV_W49_num","ANONYMOUS1GOV_W49_num"
]

# --- LOAD ---
df = pd.read_csv(DATA_PATH)

# Keep only columns that exist (avoid KeyErrors if some are missing)
existing_cols = [c for c in selected_cols if c in df.columns]
missing_in_file = sorted(set(selected_cols) - set(existing_cols))
if missing_in_file:
    print("Warning: these selected columns were not found in the dataset and will be skipped:")
    for c in missing_in_file: print("  -", c)

data = df[existing_cols].copy()

# --- 1) QUESTION-LEVEL RESPONSE COVERAGE ---
# For each question/column: count non-null, null, response rate
q_summary = []
n_rows = len(data)
for col in existing_cols:
    n_nonnull = data[col].notna().sum()
    n_null = n_rows - n_nonnull
    response_rate = (n_nonnull / n_rows) * 100 if n_rows else 0.0
    # also count #unique non-null values (helps see spreads)
    nunique_nonnull = data[col].dropna().nunique()
    q_summary.append([col, n_nonnull, n_null, response_rate, nunique_nonnull])

q_summary = pd.DataFrame(q_summary, columns=[
    "Question", "n_nonnull", "n_null", "response_rate_pct", "n_unique_nonnull"
]).sort_values(["response_rate_pct","Question"], ascending=[False, True])

q_summary.to_csv(os.path.join(OUTPUT_DIR, "question_response_summary.csv"), index=False)

print("\n== Question-level response coverage ==")
print(q_summary.to_string(index=False))

# Flags for quick inspection
low_resp = q_summary[q_summary["response_rate_pct"] < 50]
if not low_resp.empty:
    print("\nQuestions with response rate < 50%:")
    print(low_resp[["Question","response_rate_pct"]].to_string(index=False))

# --- 2) RESPONDENT-LEVEL COVERAGE (how many of the selected questions each respondent answered) ---
answered_count_per_row = data.notna().sum(axis=1)
resp_summary = answered_count_per_row.value_counts().sort_index()
resp_summary_df = resp_summary.rename_axis("n_answered").reset_index(name="n_respondents")
resp_summary_df.to_csv(os.path.join(OUTPUT_DIR, "respondent_answer_counts.csv"), index=False)

print("\n== Respondent-level: number of selected questions answered ==")
print(resp_summary_df.to_string(index=False))

print("\nBasic stats on #answered per respondent:")
print(pd.Series(answered_count_per_row).describe().to_string())

# Optional: histogram (comment out if not needed)
plt.figure(figsize=(8,5))
plt.hist(answered_count_per_row, bins=range(0, len(existing_cols)+2), align="left")
plt.title("Distribution of answered questions per respondent (selected columns)")
plt.xlabel("# of selected questions answered")
plt.ylabel("# of respondents")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "respondent_answer_counts_hist.png"))
plt.close()

# --- 3) ANSWER DISTRIBUTIONS PER QUESTION ---
# For each question, save a CSV with value counts (including NaN)
dist_dir = os.path.join(OUTPUT_DIR, "answer_distributions_per_question")
os.makedirs(dist_dir, exist_ok=True)

all_topK = []  # optional: collect top categories for quick glance

for col in existing_cols:
    vc = data[col].value_counts(dropna=False).sort_values(ascending=False)
    # Put NaN into a readable label for CSV
    labels = ["<NA>" if pd.isna(idx) else idx for idx in vc.index]
    out = pd.DataFrame({"answer": labels, "count": vc.values})
    out["pct"] = (out["count"] / n_rows * 100).round(2)
    out.to_csv(os.path.join(dist_dir, f"{col}_value_counts.csv"), index=False)

    # keep a top-5 snapshot
    snap = out.head(5).copy()
    snap.insert(0, "Question", col)
    all_topK.append(snap)

if all_topK:
    topK_df = pd.concat(all_topK, ignore_index=True)
    topK_df.to_csv(os.path.join(OUTPUT_DIR, "top5_value_counts_by_question.csv"), index=False)

print(f"\nSaved outputs to: {OUTPUT_DIR}/")
print(" - question_response_summary.csv")
print(" - respondent_answer_counts.csv (+ histogram PNG)")
print(" - answer_distributions_per_question/ (one CSV per question)")
print(" - top5_value_counts_by_question.csv")



== Question-level response coverage ==
             Question  n_nonnull  n_null  response_rate_pct  n_unique_nonnull
    CONTROLCO_W49_num       2136    2133          50.035137                 4
    CONCERNCO_W49_num       2135    2134          50.011712                 4
 UNDERSTANDCO_W49_num       2130    2139          49.894589                 4
 ANONYMOUS1CO_W49_num       2129    2140          49.871164                 2
UNDERSTANDGOV_W49_num       2126    2143          49.800890                 4
   CONCERNGOV_W49_num       2122    2147          49.707191                 4
ANONYMOUS1GOV_W49_num       2114    2155          49.519794                 2
     POSNEGCO_W49_num       2104    2165          49.285547                 2
    POSNEGGOV_W49_num       2061    2208          48.278285                 2
   CONTROLGOV_W49_num       2060    2209          48.254861                 3
    TRACKCO1a_W49_num       2053    2216          48.090888                 4
   TRACKGOV1b_W49_num   